# Industrial Machine Failure Prediction
**MCA PBL Project** · AI4I 2020 Predictive Maintenance Dataset

Goal: predict whether a machine will fail (`Machine failure` = 1) from its operating conditions and sensor readings, by training, comparing, cross-validating and tuning three classifiers (Logistic Regression, Decision Tree, Random Forest), then saving the best one for predicting new machine data.

> Place `ai4i2020.csv` in the same folder as this notebook before running.

## 0. Imports and settings

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, RocCurveDisplay)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Understand and load the dataset

| Column | Meaning |
|---|---|
| `UDI` | Row index (identifier) |
| `Product ID` | Product identifier (identifier) |
| `Type` | Machine quality category: **L**ow, **M**edium, **H**igh |
| `Air temperature [K]` | Temperature of the surrounding air |
| `Process temperature [K]` | Temperature during machine operation |
| `Rotational speed [rpm]` | Machine rotation speed |
| `Torque [Nm]` | Rotational force |
| `Tool wear [min]` | Wear accumulated by the tool |
| `Machine failure` | **Target**: 1 = failed, 0 = no failure |
| `TWF, HDF, PWF, OSF, RNF` | Failure-mode indicators (tool wear, heat dissipation, power, overstrain, random). These are *derived from the target*, so they must not be used as inputs. |

In [ ]:
df = pd.read_csv("ai4i2020.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Shorter, code-friendly column names
df = df.rename(columns={
    "Air temperature [K]": "air_temp",
    "Process temperature [K]": "process_temp",
    "Rotational speed [rpm]": "rot_speed",
    "Torque [Nm]": "torque",
    "Tool wear [min]": "tool_wear",
    "Machine failure": "failure",
})
df.columns.tolist()

## 2. Exploratory data analysis (EDA)

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate Product IDs:", df["Product ID"].duplicated().sum())

In [ ]:
df[["air_temp", "process_temp", "rot_speed", "torque", "tool_wear"]].describe().T

In [ ]:
counts = df["failure"].value_counts().rename({0: "No failure", 1: "Failure"})
print(counts)
print(f"\nFailure rate: {df['failure'].mean():.2%}  (highly imbalanced)")

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=counts.index, y=counts.values, palette=["#4c9be8", "#e8654c"], ax=ax)
for i, v in enumerate(counts.values):
    ax.text(i, v + 80, str(v), ha="center")
ax.set_title("Failed vs non-failed machines")
ax.set_ylabel("Count")
plt.show()

In [ ]:
num_cols = ["air_temp", "process_temp", "rot_speed", "torque", "tool_wear"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(data=df, x=col, hue="failure", bins=40, stat="density",
                 common_norm=False, element="step", ax=ax)
    ax.set_title(f"{col} by failure status")
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x="failure", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

df.groupby("failure")[num_cols].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

type_rate = df.groupby("Type")["failure"].agg(["count", "sum", "mean"])
type_rate.columns = ["machines", "failures", "failure_rate"]
print(type_rate)
sns.barplot(x=type_rate.index, y=type_rate["failure_rate"] * 100, ax=axes[0])
axes[0].set_title("Failure rate by machine Type (%)")
axes[0].set_ylabel("% failed")

sns.heatmap(df[num_cols + ["failure"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Physics-based relationships: torque vs speed, coloured by failure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="rot_speed", y="torque", hue="failure", alpha=0.6,
                palette={0: "#9bbcd9", 1: "#d62728"}, ax=axes[0])
axes[0].set_title("Torque vs rotational speed")
sns.scatterplot(data=df, x="tool_wear", y="torque", hue="failure", alpha=0.6,
                palette={0: "#9bbcd9", 1: "#d62728"}, ax=axes[1])
axes[1].set_title("Torque vs tool wear")
plt.tight_layout()
plt.show()

**EDA takeaways** (check these against your own plots):
- No missing values and no duplicate rows, so little cleaning is needed.
- Only about 3.4% of machines fail, so the data is heavily imbalanced. Accuracy alone is misleading (predicting "no failure" every time already gives ~96.6%), so we focus on **recall, precision and F1 for the failure class**.
- Failed machines tend to show higher torque, lower rotational speed and higher tool wear.

## 3. Prepare the data

- **Target (`y`)**: `failure`
- **Features (`X`)**: `Type`, `air_temp`, `process_temp`, `rot_speed`, `torque`, `tool_wear`
- **Dropped**: `UDI`, `Product ID` (identifiers) and `TWF, HDF, PWF, OSF, RNF` (failure-mode flags, which are leakage: they are only known after a failure has happened and `Machine failure` is basically their logical OR).
- **Preprocessing** (inside a `Pipeline`, so it is fitted on training folds only): one-hot encode `Type`, standard-scale numeric columns.

In [ ]:
# Proof of leakage: the failure-mode flags almost fully determine the target
mode_cols = ["TWF", "HDF", "PWF", "OSF", "RNF"]
any_mode = (df[mode_cols].sum(axis=1) > 0).astype(int)
print("Rows where failure == any failure-mode flag:", (any_mode == df["failure"]).mean().round(4))
print(pd.crosstab(df["failure"], any_mode, rownames=["failure"], colnames=["any mode flag"]))

In [ ]:
cat_features = ["Type"]
num_features = ["air_temp", "process_temp", "rot_speed", "torque", "tool_wear"]
features = cat_features + num_features

X = df[features].copy()
y = df["failure"].copy()
print("X shape:", X.shape, "| y shape:", y.shape)
X.head()

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

## 4. Split the dataset (80% train / 20% test, stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} rows | failure rate {y_train.mean():.2%}  ({y_train.sum()} failures)")
print(f"Test : {X_test.shape[0]} rows | failure rate {y_test.mean():.2%}  ({y_test.sum()} failures)")

## 5. Train three classification models

Each model is wrapped with the preprocessor in one `Pipeline`. This keeps scaling/encoding consistent and avoids leaking test information. These baseline models use default settings (tuned versions come in step 8).

In [ ]:
def make_pipeline(model):
    return Pipeline([("prep", preprocessor), ("model", model)])

models = {
    "Logistic Regression": make_pipeline(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "Decision Tree":       make_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
    "Random Forest":       make_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"Trained: {name}")

## 6. Evaluate and compare the models

Precision, recall and F1 are reported for the **failure class (1)**.

In [ ]:
def evaluate(pipe, X_, y_):
    pred = pipe.predict(X_)
    proba = pipe.predict_proba(X_)[:, 1]
    return {
        "Accuracy":  accuracy_score(y_, pred),
        "Precision": precision_score(y_, pred, zero_division=0),
        "Recall":    recall_score(y_, pred),
        "F1-score":  f1_score(y_, pred),
        "ROC-AUC":   roc_auc_score(y_, proba),
    }

baseline_results = pd.DataFrame({n: evaluate(p, X_test, y_test) for n, p in models.items()}).T
baseline_results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (name, pipe) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["No failure", "Failure"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.suptitle("Baseline models: confusion matrices (test set)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for name, pipe in models.items():
    print("=" * 60, "\n", name, "\n", "=" * 60)
    print(classification_report(y_test, pipe.predict(X_test), target_names=["No failure", "Failure"]))

## 7. 5-fold cross-validation (on training data)

Stratified 5-fold CV shows how consistently each model performs on different splits. F1 (failure class) is the main score, with recall and accuracy shown too.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows, cv_f1_scores = [], {}
for name, pipe in models.items():
    f1s  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    rec  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="recall", n_jobs=-1)
    acc  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    cv_f1_scores[name] = f1s
    print(f"{name}\n  F1 per fold: {np.round(f1s, 4)}")
    cv_rows.append({"Model": name,
                    "CV F1 (mean)": f1s.mean(), "CV F1 (std)": f1s.std(),
                    "CV Recall (mean)": rec.mean(), "CV Accuracy (mean)": acc.mean()})

cv_results = pd.DataFrame(cv_rows).set_index("Model")
cv_results.round(4)

In [ ]:
pd.DataFrame(cv_f1_scores, index=[f"Fold {i}" for i in range(1, 6)]).T.plot(
    kind="bar", figsize=(9, 4.5), rot=0)
plt.title("5-fold cross-validation F1-score (failure class)")
plt.ylabel("F1-score")
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## 8. Hyperparameter tuning with GridSearchCV

Scoring metric: **F1-score** (balances precision and recall for the rare failure class). `class_weight="balanced"` is included in the grids so the models can give more importance to failures.

In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__class_weight": [None, "balanced"],
    },
    "Decision Tree": {
        "model__max_depth": [3, 5, 7, 10, None],
        "model__min_samples_leaf": [1, 5, 10, 20],
        "model__criterion": ["gini", "entropy"],
        "model__class_weight": [None, "balanced"],
    },
    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [1, 2, 5],
        "model__class_weight": [None, "balanced"],
    },
}

tuned, tune_rows = {}, []
for name, pipe in models.items():
    gs = GridSearchCV(pipe, param_grids[name], scoring="f1", cv=cv, n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    tuned[name] = gs
    best = {k.replace("model__", ""): v for k, v in gs.best_params_.items()}
    print(f"{name}\n  Best params: {best}\n  Best CV F1 : {gs.best_score_:.4f}\n")
    tune_rows.append({"Model": name, "Best CV F1": gs.best_score_, "Best parameters": best})

tuning_results = pd.DataFrame(tune_rows).set_index("Model")
tuning_results

## 9. Select the best model

Compare the tuned models using CV F1 (used for selection), then evaluate tuned models once on the untouched test set. The final choice is based on CV so that the test set stays a fair, unbiased final check.

In [ ]:
compare = []
for name, gs in tuned.items():
    m = evaluate(gs.best_estimator_, X_test, y_test)
    m["Best CV F1"] = gs.best_score_
    compare.append(pd.Series(m, name=name))
tuned_comparison = pd.DataFrame(compare)[["Best CV F1", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]]
tuned_comparison.round(4)

In [ ]:
best_name = max(tuned, key=lambda n: tuned[n].best_score_)
best_model = tuned[best_name].best_estimator_
print(f"Selected model: {best_name}")
print("Best parameters:", {k.replace('model__', ''): v for k, v in tuned[best_name].best_params_.items()})

In [ ]:
y_pred = best_model.predict(X_test)
final_metrics = evaluate(best_model, X_test, y_test)
print(f"FINAL TEST METRICS: {best_name}")
for k, v in final_metrics.items():
    print(f"  {k:<10}: {v:.4f}")

print()
print(classification_report(y_test, y_pred, target_names=["No failure", "Failure"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No failure", "Failure"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Confusion matrix: {best_name}")

cm_norm = confusion_matrix(y_test, y_pred, normalize="true")
ConfusionMatrixDisplay(cm_norm, display_labels=["No failure", "Failure"]).plot(
    ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f")
axes[1].set_title("Normalised (by actual class)")

RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=axes[2])
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[2].set_title("ROC curve")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Caught {tp} of {tp + fn} real failures; missed {fn}; {fp} false alarms.")

In [ ]:
# Which inputs matter most? (works for tree-based models and Logistic Regression)
final_est = best_model.named_steps["model"]
feat_names = best_model.named_steps["prep"].get_feature_names_out()
feat_names = [f.split("__")[-1] for f in feat_names]

if hasattr(final_est, "feature_importances_"):
    imp = pd.Series(final_est.feature_importances_, index=feat_names).sort_values()
    title = "Feature importance"
else:
    imp = pd.Series(final_est.coef_[0], index=feat_names).sort_values()
    title = "Logistic Regression coefficients (standardised inputs)"

imp.plot(kind="barh", figsize=(7, 4))
plt.title(title)
plt.tight_layout()
plt.show()

## 10 & 11. Save the model and predict new, unseen machine data

The saved file contains the **whole pipeline** (encoding + scaling + model), so it can predict raw inputs without retraining.

In [ ]:
MODEL_PATH = "machine_failure_model.joblib"
joblib.dump({"pipeline": best_model, "features": features, "model_name": best_name}, MODEL_PATH)
print("Saved:", MODEL_PATH)

# Reload to confirm it works independently of the training objects
artifact = joblib.load(MODEL_PATH)
loaded_pipeline = artifact["pipeline"]
print("Reloaded model:", artifact["model_name"])

In [ ]:
def predict_machine(machine_type, air_temp, process_temp, rot_speed, torque, tool_wear,
                    pipeline=None, threshold=0.5):
    """Predict failure for one machine. machine_type must be 'L', 'M' or 'H'."""
    pipeline = pipeline or loaded_pipeline
    machine_type = str(machine_type).strip().upper()
    if machine_type not in {"L", "M", "H"}:
        raise ValueError("Type must be L, M or H")
    row = pd.DataFrame([{
        "Type": machine_type, "air_temp": air_temp, "process_temp": process_temp,
        "rot_speed": rot_speed, "torque": torque, "tool_wear": tool_wear,
    }])[features]
    prob = pipeline.predict_proba(row)[0, 1]
    label = "Yes" if prob >= threshold else "No"
    return label, prob

def show_prediction(*args, **kwargs):
    label, prob = predict_machine(*args, **kwargs)
    print("=" * 40)
    print(" Machine Failure Prediction System")
    print("=" * 40)
    print(f" Machine Failure : {label}")
    print(f" Failure probability : {prob:.1%}")
    print("=" * 40)

# Example 1: normal-looking operating conditions
show_prediction("M", air_temp=300, process_temp=310, rot_speed=1500, torque=40, tool_wear=100)
# Example 2: high torque, low speed, heavily worn tool
show_prediction("L", air_temp=302, process_temp=311, rot_speed=1300, torque=65, tool_wear=230)

In [ ]:
# Sanity check on a few real rows from the dataset (using their true labels)
sample = df.sample(5, random_state=1)
for _, r in sample.iterrows():
    label, prob = predict_machine(r["Type"], r["air_temp"], r["process_temp"],
                                  r["rot_speed"], r["torque"], r["tool_wear"])
    print(f"Actual: {'Yes' if r['failure'] else 'No':<3} | Predicted: {label:<3} | P(failure) = {prob:.1%}")

### Interactive command-line interface

Run the cell below and type values when prompted. Type `q` at any prompt to quit.

In [ ]:
def run_cli():
    print("Machine Failure Prediction System  (type q to quit)\n")
    while True:
        try:
            t = input("Machine Type (L/M/H): ").strip()
            if t.lower() == "q": break
            vals = []
            for prompt in ["Air temperature [K]", "Process temperature [K]",
                           "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]"]:
                v = input(f"{prompt}: ").strip()
                if v.lower() == "q": return
                vals.append(float(v))
            show_prediction(t, *vals)
        except ValueError as e:
            print(f"Invalid input ({e}). Please try again.\n")
            continue
        if input("Predict another? (y/n): ").strip().lower() != "y":
            break

run_cli()

### Standalone script (optional)

To run predictions without this notebook, save the following as `predict.py` next to `machine_failure_model.joblib`:

```python
import joblib, pandas as pd
art = joblib.load("machine_failure_model.joblib")
pipe, feats = art["pipeline"], art["features"]

t  = input("Machine Type (L/M/H): ").strip().upper()
at = float(input("Air temperature [K]: "))
pt = float(input("Process temperature [K]: "))
rs = float(input("Rotational speed [rpm]: "))
tq = float(input("Torque [Nm]: "))
tw = float(input("Tool wear [min]: "))

row = pd.DataFrame([dict(Type=t, air_temp=at, process_temp=pt, rot_speed=rs, torque=tq, tool_wear=tw)])[feats]
p = pipe.predict_proba(row)[0, 1]
print("Machine Failure:", "Yes" if p >= 0.5 else "No", f"(probability {p:.1%})")
```

## Summary

In [ ]:
print("Models trained        : Logistic Regression, Decision Tree, Random Forest")
print("Baseline comparison   :"); display(baseline_results.round(4))
print("\n5-fold CV results    :"); display(cv_results.round(4))
print("\nTuned hyperparameters:"); display(tuning_results)
print(f"\nSelected model: {best_name}")
print("Final test metrics:", {k: round(v, 4) for k, v in final_metrics.items()})